# Phase 1: Realistic Vision v5.1 Deepfake Dataset Generator
**Dự án Topic 11 (Nhóm 4)**: Tạo bộ dữ liệu khuôn mặt giả bằng Stable Diffusion phục vụ phát hiện Deepfake.

### Quy trình thực hiện:
- **Task 1 (Inpainting)**: Sửa vùng miệng (nụ cười) hoặc vùng mắt (kính râm) trên khuôn mặt thật.
- **Task 2 (Img2Img)**: Biến đổi cấu trúc da và ánh sáng từ ảnh thật (`strength = 0.55`).
- **Task 3 (Text2Img)**: Sinh khuôn mặt độc lập hoàn toàn mới theo các prompt nhân khẩu học cân bằng.

> **Lưu ý**: Chạy **NUM_SAMPLES = 10** ở Cell 3 trước để test thử trong ~1.5 phút. Sau khi kiểm tra ảnh nét, đổi `NUM_SAMPLES = 2000` để chạy full 6.000 ảnh.

### Bước 1: Cài đặt các thư viện cần thiết

In [ ]:
# Nang cap va dong bo phien ban giua diffusers va transformers tren Kaggle
!pip install -q diffusers transformers accelerate

### Bước 2: Khai báo thư viện & Kiểm tra GPU

In [ ]:
# ==============================================================================
# BẢN VÁ TƯƠNG THÍCH CHO KAGGLE PYTHON 3.13
# ==============================================================================
import transformers
import transformers.utils

# 1. Vá lỗi FLAX_WEIGHTS_NAME
if not hasattr(transformers.utils, 'FLAX_WEIGHTS_NAME'):
    transformers.utils.FLAX_WEIGHTS_NAME = 'flax_model.msgpack'

# 2. Vá lỗi CLIPImageProcessor do lazy-loading bị lỗi trên Python 3.13
try:
    from transformers import CLIPImageProcessor
except Exception:
    class DummyCLIPImageProcessor:
        @classmethod
        def from_pretrained(cls, *args, **kwargs):
            return cls()
    transformers.CLIPImageProcessor = DummyCLIPImageProcessor

# ==============================================================================
# KHAI BÁO THƯ VIỆN & KIỂM TRA GPU
# ==============================================================================
import glob
import itertools
import json
import os
import random
import shutil
import gc
from pathlib import Path
from PIL import Image, ImageDraw
import torch
import matplotlib.pyplot as plt
from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionImg2ImgPipeline,
    StableDiffusionInpaintPipeline,
    DPMSolverMultistepScheduler,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Không có GPU (Đang dùng CPU)'
print(f'[✓] PyTorch: {torch.__version__} | Thiết bị: {DEVICE} ({gpu_name})')

### Bước 3: Cấu hình tham số & Tự động quét Dataset ảnh thật
*(Bạn có thể chỉnh `NUM_SAMPLES` tại đây: 10 ảnh test hoặc 2000 ảnh full)*

In [ ]:
# ==============================================================================
# CẤU HÌNH THAM SỐ CHÍNH
# ==============================================================================
NUM_SAMPLES = 2000        # <-- ĐỔI THÀNH 2000 ĐỂ CHẠY FULL 6.000 ẢNH!
STEPS = 25                # 25 steps với DPM++ 2M Karras (nhanh và cực nét)
GUIDANCE_SCALE = 7.0
IMG2IMG_STRENGTH = 0.55   # 0.55 giữ nguyên cấu trúc mặt thật nhưng biến đổi texture da
SEED = 42

BASE_MODEL = 'SG161222/Realistic_Vision_V5.1_noVAE'
INPAINT_MODEL = "stable-diffusion-v1-5/stable-diffusion-inpainting"

OUTPUT_DIR = Path('/kaggle/working/data/synthetic')
DIR_INPAINT = OUTPUT_DIR / 'inpaint'
DIR_IMG2IMG = OUTPUT_DIR / 'img2img'
DIR_TEXT2IMG = OUTPUT_DIR / 'text2img'
MANIFEST_PATH = OUTPUT_DIR / 'fake_manifest.jsonl'

for d in [DIR_INPAINT, DIR_IMG2IMG, DIR_TEXT2IMG]:
    d.mkdir(parents=True, exist_ok=True)

# Tự động tìm kiếm ảnh CelebA-HQ trong /kaggle/input
candidate_files = sorted(glob.glob('/kaggle/input/**/celeba_hq/*.jpg', recursive=True))
if not candidate_files:
    candidate_files = sorted(glob.glob('/kaggle/input/**/*.jpg', recursive=True))
if not candidate_files:
    candidate_files = sorted(glob.glob('/kaggle/input/**/*.png', recursive=True))

if not candidate_files:
    raise FileNotFoundError(
        'Không tìm thấy ảnh thật nào trong /kaggle/input! Hãy kiểm tra xem bạn đã bấm + Add Input dataset chưa.'
    )

selected_real_files = candidate_files[:NUM_SAMPLES]
print(f'[✓] Đã tìm thấy {len(candidate_files)} ảnh gốc trong dataset.')
print(f'[✓] Lựa chọn: {len(selected_real_files)} ảnh để sinh dữ liệu.')
print(f'[✓] Chế độ: {"TEST NHANH 10 ẢNH (~1.5 phút)" if NUM_SAMPLES <= 10 else "CHẠY FULL TOÀN BỘ 2.000 ẢNH"}')

### Bước 4: Khởi tạo các hàm sinh Prompt, Mặt nạ (Mask) & Lưu nhãn

In [ ]:
def generate_prompts(num_prompts: int, seed: int = 42) -> list[str]:
    random.seed(seed)
    ages = [20, 24, 28, 32, 36, 40, 45, 50, 55, 60]
    genders = ['man', 'woman']
    ethnicities = ['Asian', 'Caucasian', 'African', 'Hispanic', 'South Asian']
    expressions = [
        'gentle natural smile',
        'neutral serious expression',
        'thoughtful calm gaze',
        'subtle warm smile',
        'confident relaxed expression',
    ]
    lightings = [
        'studio softbox lighting',
        'cinematic warm side lighting',
        'diffuse natural window lighting',
        'soft rim lighting with subtle shadows',
    ]
    combos = list(itertools.product(ages, genders, ethnicities, expressions, lightings))
    random.shuffle(combos)
    prompts = []
    for i in range(num_prompts):
        age, gender, ethnicity, expression, lighting = combos[i % len(combos)]
        p = (
            f'RAW photo, a close-up portrait of a {age}-year-old {ethnicity} {gender}, '
            f'{expression}, neutral grey studio backdrop, {lighting}, '
            f'detailed skin texture, visible pores, 8k uhd, dslr, high quality'
        )
        prompts.append(p)
    return prompts

NEGATIVE_PROMPT = (
    'deformed, bad eyes, bad anatomy, bad hands, poorly drawn face, mutation, '
    'mutated, ugly, disgusting, cartoon, 3d, anime, airbrushed, oversaturated, '
    'blurry, low quality, watermark, text, signature'
)

def create_mask(size: tuple[int, int], region: str) -> Image.Image:
    w, h = size
    mask = Image.new('L', (w, h), 0)
    draw = ImageDraw.Draw(mask)
    if region == 'mouth':
        box = [int(w * 0.30), int(h * 0.60), int(w * 0.70), int(h * 0.85)]
    else:  # eyes
        box = [int(w * 0.20), int(h * 0.25), int(w * 0.80), int(h * 0.52)]
    draw.ellipse(box, fill=255)
    return mask

def append_manifest(record: dict):
    with open(MANIFEST_PATH, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print('[✓] Đã khởi tạo các hàm tiện ích.')

### Bước 5: Chạy Task 1 — Inpainting (50% vùng miệng + 50% vùng mắt)

In [ ]:
print('=' * 60)
print(f'[*] [1/3] BẮT ĐẦU TASK INPAINTING ({len(selected_real_files)} ảnh)...')
print('=' * 60)

pipe_inpaint = StableDiffusionInpaintPipeline.from_pretrained(
    INPAINT_MODEL,
    torch_dtype=DTYPE,
    safety_checker=None,
).to(DEVICE)
pipe_inpaint.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe_inpaint.scheduler.config, use_karras_sigmas=True
)

for idx, file_path in enumerate(selected_real_files):
    stem = Path(file_path).stem
    out_file = DIR_INPAINT / f'{stem}_inpaint.png'
    if out_file.exists():
        continue
    
    init_img = Image.open(file_path).convert('RGB').resize((512, 512), Image.Resampling.LANCZOS)
    region = 'mouth' if (idx % 2 == 0) else 'eyes'
    mask_img = create_mask(init_img.size, region)
    
    if region == 'mouth':
        prompt = 'a natural smiling mouth, realistic lips and skin texture, photorealistic, 8k uhd'
    else:
        prompt = 'wearing stylish black sunglasses, natural skin reflections, photorealistic, 8k uhd'
        
    generator = torch.Generator(device=DEVICE).manual_seed(SEED + idx)
    result = pipe_inpaint(
        prompt=prompt,
        image=init_img,
        mask_image=mask_img,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
    ).images[0]
    
    result.save(out_file)
    append_manifest({
        'image_path': str(out_file.relative_to(OUTPUT_DIR.parent.parent)),
        'task': 'inpaint',
        'sub_region': region,
        'source_real_image': Path(file_path).name,
        'prompt': prompt,
        'seed': SEED + idx,
    })
    # Thay vì in liên tục, cứ mỗi 50 ảnh in tiến độ 1 lần:
    if (idx + 1) % 50 == 0 or (idx + 1) == len(selected_real_files):
        print(f"  [Tiến độ: {idx+1}/{len(selected_real_files)}] -> {out_file.name}")

# Giải phóng VRAM cho model kế tiếp
del pipe_inpaint
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print('[✓] Task Inpaint hoàn tất & đã giải phóng VRAM.')

### Bước 6: Chạy Task 2 — Img2Img (Biến đổi kết cấu da & Ánh sáng)

In [ ]:
print('=' * 60)
print(f'[*] [2/3] BẮT ĐẦU TASK IMG2IMG ({len(selected_real_files)} ảnh)...')
print('=' * 60)

pipe_img2img = StableDiffusionImg2ImgPipeline.from_pretrained(
    BASE_MODEL,
    torch_dtype=DTYPE,
    safety_checker=None,
).to(DEVICE)
pipe_img2img.scheduler = DPMSolverMultistepScheduler.from_config(pipe_img2img.scheduler.config, algorithm_type="dpmsolver++", use_karras_sigmas=True)

prompt_img2img = 'RAW photo, a close-up portrait of an adult person, natural skin texture, studio lighting, photorealistic, 8k uhd, dslr'

for idx, file_path in enumerate(selected_real_files):
    stem = Path(file_path).stem
    out_file = DIR_IMG2IMG / f'{stem}_img2img.png'
    if out_file.exists():
        continue
        
    init_img = Image.open(file_path).convert('RGB').resize((512, 512), Image.Resampling.LANCZOS)
    generator = torch.Generator(device=DEVICE).manual_seed(SEED + idx)
    result = pipe_img2img(
        prompt=prompt_img2img,
        image=init_img,
        strength=IMG2IMG_STRENGTH,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
    ).images[0]
    
    result.save(out_file)
    append_manifest({
        'image_path': str(out_file.relative_to(OUTPUT_DIR.parent.parent)),
        'task': 'img2img',
        'source_real_image': Path(file_path).name,
        'prompt': prompt_img2img,
        'seed': SEED + idx,
        'strength': IMG2IMG_STRENGTH,
    })
    if (idx + 1) % 5 == 0 or (idx + 1) == len(selected_real_files):
        print(f'  [Img2Img: {idx+1}/{len(selected_real_files)}] -> {out_file.name}')

# Giải phóng VRAM
del pipe_img2img
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print('[✓] Task Img2Img hoàn tất & đã giải phóng VRAM.')

### Bước 7: Chạy Task 3 — Text2Img (Sinh khuôn mặt độc lập hoàn toàn mới)

In [ ]:
print('=' * 60)
print(f'[*] [3/3] BẮT ĐẦU TASK TEXT2IMG ({len(selected_real_files)} ảnh)...')
print('=' * 60)

pipe_text2img = StableDiffusionPipeline.from_pretrained(
    BASE_MODEL,
    torch_dtype=DTYPE,
    safety_checker=None,
).to(DEVICE)
pipe_text2img.scheduler = DPMSolverMultistepScheduler.from_config(pipe_text2img.scheduler.config, algorithm_type="dpmsolver++", use_karras_sigmas=True)

t2i_prompts = generate_prompts(len(selected_real_files), seed=SEED)

for idx, (file_path, prompt) in enumerate(zip(selected_real_files, t2i_prompts)):
    stem = Path(file_path).stem
    out_file = DIR_TEXT2IMG / f'{stem}_text2img.png'
    if out_file.exists():
        continue
        
    generator = torch.Generator(device=DEVICE).manual_seed(SEED + idx)
    result = pipe_text2img(
        prompt=prompt,
        height=512,
        width=512,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
    ).images[0]
    
    result.save(out_file)
    append_manifest({
        'image_path': str(out_file.relative_to(OUTPUT_DIR.parent.parent)),
        'task': 'text2img',
        'source_real_image': None,
        'prompt': prompt,
        'seed': SEED + idx,
    })
    if (idx + 1) % 5 == 0 or (idx + 1) == len(selected_real_files):
        print(f'  [Text2Img: {idx+1}/{len(selected_real_files)}] -> {out_file.name}')

# Giải phóng VRAM
del pipe_text2img
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print('[✓] Task Text2Img hoàn tất & đã giải phóng VRAM.')

### Bước 8: Đóng gói ZIP & Xem ảnh kiểm tra trực quan (Gallery)

In [ ]:
zip_path = '/kaggle/working/synthetic_face_dataset.zip'
print(f'[*] Đang nén toàn bộ kết quả vào {zip_path}...')
shutil.make_archive('/kaggle/working/synthetic_face_dataset', 'zip', root_dir='/kaggle/working/data/synthetic')
print(f'[✓] Đã tạo thành công file: {zip_path} ({os.path.getsize(zip_path) / 1024 / 1024:.2f} MB)\n')

# Hiển thị ảnh đại diện 3 task
inpaint_sample = next(DIR_INPAINT.glob('*.png'), None)
img2img_sample = next(DIR_IMG2IMG.glob('*.png'), None)
text2img_sample = next(DIR_TEXT2IMG.glob('*.png'), None)

if inpaint_sample and img2img_sample and text2img_sample:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(Image.open(inpaint_sample))
    axes[0].set_title(f'Inpaint: {inpaint_sample.name}', fontsize=12)
    axes[0].axis('off')
    
    axes[1].imshow(Image.open(img2img_sample))
    axes[1].set_title(f'Img2Img: {img2img_sample.name}', fontsize=12)
    axes[1].axis('off')
    
    axes[2].imshow(Image.open(text2img_sample))
    axes[2].set_title(f'Text2Img: {text2img_sample.name}', fontsize=12)
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

total_gen = len(list(DIR_INPAINT.glob('*.png'))) + len(list(DIR_IMG2IMG.glob('*.png'))) + len(list(DIR_TEXT2IMG.glob('*.png')))
print('=' * 65)
print(f'[✓] HOÀN TẤT! ĐÃ TẠO THÀNH CÔNG TỔNG CỘNG: {total_gen} ẢNH GIẢ.')
print('• Tải file dataset: Nhìn sang cột bên phải (Output) -> bấm tải synthetic_face_dataset.zip')
print('• Muốn chạy Full 6.000 ảnh: Đổi NUM_SAMPLES = 2000 ở Bước 3 rồi bấm Run!')
print('=' * 65)